# Drift Detection — Final (bugs fixed)

**Bugs fixed in this version:**
1. Boolean columns (e.g. `is_eligible`) were silently skipped — `np.issubdtype(bool_dtype, np.number)` is `False` in NumPy, and bool also isn't `object`/`CategoricalDtype`, so they fell through to `continue` and were never drift-checked. Now booleans are explicitly routed to the categorical (chi-square) test.
2. The drift report was only ever printed to the cell output — nothing was saved anywhere, so no automated retraining trigger could ever consume it. Now the report is saved to JSON with a clear overall verdict.
3. No multiple-testing correction — testing every column independently at alpha=0.05 inflates the false-positive drift-alert rate as the number of columns grows. A Bonferroni correction is now applied.
4. Hardcoded local Windows paths replaced with relative paths.


## 1. Load baseline and current data

**Industry rule:** the baseline must be the data the model was actually trained on, and "current" must be processed through the exact same pipeline as the baseline. Comparing data at different pipeline stages (e.g. raw vs. cleaned) will show fake drift that has nothing to do with the real world changing.

In [1]:
import numpy as np
import pandas as pd
import json
from scipy.stats import ks_2samp, chi2_contingency

train_df = pd.read_csv("../data/processed/train_df.csv")
test_df = pd.read_csv("../data/processed/test_df.csv")


## 2. Drift detection class

**Industry rule:** always separate the statistical test by feature type — Kolmogorov-Smirnov for continuous features, chi-square for categorical/boolean features. Using the wrong test for a feature type gives a meaningless p-value.

In [2]:
class StatisticalDriftDetection:
    def __init__(self, threshold=0.05, apply_multiple_testing_correction=True):
        # Industry standard alpha is usually 0.05, but with many columns tested
        # independently, a Bonferroni correction should be applied to control the
        # overall false-positive rate (see detect_feature_drift below).
        self.threshold = threshold
        self.apply_multiple_testing_correction = apply_multiple_testing_correction

    # ======== KOLMOGOROV-SMIRNOV TEST (continuous features) ========
    def ks_test(self, baseline_data, current_data, alpha):
        statistic, p_value = ks_2samp(baseline_data, current_data)
        return {
            'test': 'Kolmogorov-Smirnov',
            'p_value': round(float(p_value), 5),
            'drift_detected': bool(p_value < alpha),
            'interpretation': 'DRIFT DETECTED' if p_value < alpha else 'No drift',
        }

    # ======== CHI-SQUARE TEST (categorical AND boolean features) ========
    def chi_square_test(self, baseline_cat, current_cat, alpha):
        # Normalize counts to percentages to handle different sample sizes safely.
        b_counts = baseline_cat.value_counts(normalize=True)
        c_counts = current_cat.value_counts(normalize=True)

        all_categories = set(b_counts.index) | set(c_counts.index)
        # Safe filling with a tiny value instead of 0 to avoid chi2 math errors.
        b_counts = b_counts.reindex(all_categories, fill_value=1e-5)
        c_counts = c_counts.reindex(all_categories, fill_value=1e-5)

        try:
            dummy_population = 1000
            observed = np.array([b_counts.values * dummy_population, c_counts.values * dummy_population])
            chi2, p_value, dof, expected = chi2_contingency(observed)
            drift_detected = bool(p_value < alpha)
            interpretation = 'DRIFT DETECTED' if drift_detected else 'No Drift'
        except Exception:
            p_value = None
            drift_detected = True
            interpretation = 'DRIFT DETECTED (test error / new categories)'

        return {
            'test': 'Chi-Square',
            'p_value': round(float(p_value), 5) if p_value is not None else None,
            'drift_detected': drift_detected,
            'interpretation': interpretation,
        }

    # ======== FEATURE DRIFT DETECTION (all columns) ========
    def detect_feature_drift(self, baseline_data, current_data):
        drift_report = {}
        checkable_cols = [c for c in baseline_data.columns if c in current_data.columns]

        # Bonferroni correction: dividing alpha by the number of tests keeps the
        # overall false-positive rate across all columns close to the original 0.05,
        # instead of it compounding upward as more columns are tested.
        n_tests = max(len(checkable_cols), 1)
        alpha = self.threshold / n_tests if self.apply_multiple_testing_correction else self.threshold

        for col in checkable_cols:
            baseline = baseline_data[col].dropna()
            current = current_data[col].dropna()
            dtype = baseline_data[col].dtype

            # FIXED: bool must be checked BEFORE the numeric check, and explicitly
            # routed to chi-square — np.issubdtype(bool, np.number) is False, and
            # bool is also not 'object'/CategoricalDtype, so it was previously
            # silently skipped entirely (never drift-checked).
            if pd.api.types.is_bool_dtype(dtype):
                result = self.chi_square_test(baseline, current, alpha)
            elif np.issubdtype(dtype, np.number):
                result = self.ks_test(baseline.values, current.values, alpha)
            elif dtype == 'object' or isinstance(dtype, pd.CategoricalDtype):
                result = self.chi_square_test(baseline, current, alpha)
            else:
                # Safe skip for datetime, free text, or ID columns.
                continue

            drift_report[col] = result

        return pd.DataFrame.from_dict(drift_report, orient='index'), alpha


## 3. Run drift check

**Industry rule:** always test the detector against a *known* injected shift before trusting it on real data — if it can't catch an obvious, deliberately introduced shift, it won't catch a subtle real one either.

In [3]:
result_class = StatisticalDriftDetection(threshold=0.05, apply_multiple_testing_correction=True)

baseline_data = train_df

# Sanity-check simulation: deliberately shift Income to confirm the detector
# actually fires. Do NOT skip this step when validating a new detector version.
current_data = test_df.copy()
current_data['Income'] = current_data['Income'] * 1.15

drift_report_df, alpha_used = result_class.detect_feature_drift(baseline_data, current_data)
print(f"Alpha used per test (after correction): {alpha_used:.6f}")
drift_report_df


Alpha used per test (after correction): 0.004167


,test,p_value,drift_detected,interpretation
Age,Kolmogorov-Smirnov,0.63579,False,No drift
Income,Kolmogorov-Smirnov,0.00000,True,DRIFT DETECTED
Home_ownership,Chi-Square,0.99352,False,No Drift
Employment_length,Kolmogorov-Smirnov,0.40410,False,No drift
Loan_intent,Chi-Square,0.94272,False,No Drift
internal_credit_rating,Chi-Square,0.99959,False,No Drift
Loan_amount,Kolmogorov-Smirnov,0.77344,False,No drift
Interest_rate,Kolmogorov-Smirnov,0.68241,False,No drift
Loan_percent_income,Kolmogorov-Smirnov,0.19431,False,No drift
Default,Chi-Square,1.00000,False,No Drift


## 4. Save drift report + overall verdict (FIXED — now persisted, not just printed)

**Industry rule:** a monitoring check that isn't written anywhere cannot trigger anything downstream. This report is what `scheduled_drift_job.py` / the retraining DAG should actually read from.

In [4]:
n_drifted = int(drift_report_df['drift_detected'].sum())
n_checked = len(drift_report_df)
overall_drift_detected = n_drifted > 0

drift_summary = {
    'columns_checked': n_checked,
    'columns_drifted': n_drifted,
    'alpha_used': round(float(alpha_used), 6),
    'overall_drift_detected': overall_drift_detected,
    'drifted_columns': drift_report_df[drift_report_df['drift_detected']].index.tolist(),
    'per_column_report': drift_report_df.to_dict(orient='index'),
}

with open("../reports/drift_report.json", "w") as f:
    json.dump(drift_summary, f, indent=2)

print(f"Checked {n_checked} columns, {n_drifted} flagged as drifted.")
print(f"Overall drift detected: {overall_drift_detected}")
print("Saved to ../reports/drift_report.json")


Checked 12 columns, 1 flagged as drifted.
Overall drift detected: True
Saved to ../reports/drift_report.json
